# ⚡ ALL-IN-ONE STANDALONE DENSE GPU PIPELINE (NO BM25)
### Road to AI 2026 (R2AI) — ViBioMIR Medical Document Retrieval

> **Tài nguyên**: Chạy trên **Kaggle GPU (T4 x 2 hoặc P100)**.
> **Chiến lược**: **100% Dense Retrieval (BGE-M3 + Faiss)** — Loại bỏ hoàn toàn gánh nặng BM25 / Tokenizer / Tràn đĩa.
> **Quy trình khép kín trong 1 Notebook duy nhất**:
> 1. Tải 14 shards Parquet (1,025,647 bài viết, chỉ ~850 MB nén ZSTD) cùng bảng làm sạch (`boilerplate_lines.parquet`, `corpus_audit_slim.parquet`).
> 2. Bỏ dòng điều hướng/footer lặp theo domain + document có `verdict = drop`, rồi phân đoạn **chunk 750 ký tự** (overlap 120 ký tự) trực tiếp vào SQLite `chunks_meta.sqlite` (xóa ngay parquet sau khi xong để tiết kiệm 100% ổ đĩa).
> 3. Dùng GPU mã hóa vector 1024-dim với **`BAAI/bge-m3`** (FP16).
> 4. Xây dựng chỉ mục **`dense_index.faiss`** cực nhanh và ổn định.
> 5. Tự động đồng bộ `dense_index.faiss` và `chunks_meta.sqlite` lên Hugging Face Storage Bucket.

In [ ]:
# 1. Kiểm tra phần cứng GPU
!nvidia-smi

In [ ]:
# 2. Cài đặt các thư viện cần thiết siêu tốc với uv
!pip install -q uv
!uv pip install --system --no-cache faiss-gpu pyarrow "huggingface_hub[cli]" loguru tqdm flagembedding sentence-transformers torch

In [ ]:
# 3. Clone / Cập nhật mã nguồn dự án
import os
from pathlib import Path

REPO_URL = "https://github.com/neiths/med-doc-retrieval.git"
PROJECT_DIR = Path("/kaggle/working/med-doc-retrieval")

if not PROJECT_DIR.exists():
    print("🚀 Đang clone repository từ GitHub...")
    !git clone -b main $REPO_URL $PROJECT_DIR
else:
    print("🔄 Đang cập nhật repository...")
    %cd $PROJECT_DIR
    !git pull origin main

%cd $PROJECT_DIR
print("Thư mục làm việc hiện tại:", os.getcwd())

In [ ]:
# 4. Cấu hình Hugging Face Token (Chống lỗi Bearer rỗng)
import os
from huggingface_hub import HfApi, login

HF_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN or not HF_TOKEN.strip():
    print('⚠️ CHƯA TÌM THẤY HF_TOKEN!')
    print('👉 Cách 1 (Khuyên dùng): Vào menu Add-ons -> Secrets trên Kaggle, thêm secret tên HF_TOKEN.')
    print('👉 Cách 2: Nhập trực tiếp token Hugging Face của bạn vào bên dưới:')
    HF_TOKEN = input('Nhập HF_TOKEN: ').strip()

if not HF_TOKEN:
    raise ValueError('❌ HF_TOKEN không được để trống! Không thể tải dữ liệu từ private HF Bucket.')

os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
print(f'✅ Đã kết nối Hugging Face thành công! (Token: {HF_TOKEN[:7]}...)')

In [ ]:
# 5. Tải 14 Shards Parquet (850 MB) + bảng làm sạch từ Hugging Face Bucket
import os
import subprocess
from pathlib import Path

raw_corpus_dir = Path("data/raw/corpus")
raw_corpus_dir.mkdir(parents=True, exist_ok=True)

shard_files = []
for i in range(14):
    fname = f"corpus_part_{i:03d}.parquet"
    shard_files.append((f"corpus/{fname}", str(raw_corpus_dir / fname)))

to_download = [(r, l) for r, l in shard_files if not Path(l).exists() or Path(l).stat().st_size == 0]
if to_download:
    print(f"📥 Đang tải {len(to_download)} shards parquet từ HF Bucket...")
    api.download_bucket_files(
        bucket_id="nieths/ViBioMIR",
        files=to_download,
        token=HF_TOKEN,
    )
    print("✅ Đã tải xong 14 shards parquet!")
else:
    print("✓ Toàn bộ 14 shards parquet đã có sẵn trên ổ đĩa!")

# Bảng làm sạch: dòng boilerplate lặp theo domain + verdict audit từng document
clean_dir = Path("data/processed")
clean_dir.mkdir(parents=True, exist_ok=True)
for fname in ("boilerplate_lines.parquet", "corpus_audit_slim.parquet"):
    local_p = clean_dir / fname
    if local_p.exists() and local_p.stat().st_size > 0:
        print(f"✓ Đã có: {fname} ({local_p.stat().st_size / (1024*1024):.2f} MB)")
        continue
    print(f"📥 Đang tải {fname} từ HF Bucket...")
    subprocess.run(
        ["hf", "buckets", "cp", f"hf://buckets/nieths/ViBioMIR/corpus/{fname}", str(local_p)],
        check=True,
    )
print("✅ Bảng làm sạch đã sẵn sàng!")

In [ ]:
# 6. Cắt Chunks 750 ký tự trên dữ liệu ĐÃ LÀM SẠCH & ghi vào SQLite
import sqlite3
import pyarrow.parquet as pq
from tqdm import tqdm
from src.ingestion.boilerplate import load_table
from src.ingestion.chunker import DocumentChunker

indices_dir = Path("data/indices")
indices_dir.mkdir(parents=True, exist_ok=True)
sqlite_path = indices_dir / "chunks_meta.sqlite"

# Làm sạch 1: bỏ dòng điều hướng/footer lặp lại trong từng domain
boilerplate_path = Path("data/processed/boilerplate_lines.parquet")
boilerplate = load_table(boilerplate_path) if boilerplate_path.exists() else {}
print(
    f"🧹 Boilerplate: {len(boilerplate):,} domains, "
    f"{sum(len(v) for v in boilerplate.values()):,} dòng bị loại"
)

# Làm sạch 2: bỏ document có verdict = drop trong bảng audit
audit_path = Path("data/processed/corpus_audit_slim.parquet")
drop_ids = set()
if audit_path.exists():
    audit_tbl = pq.read_table(str(audit_path), columns=["doc_id", "verdict"])
    for did, verdict in zip(audit_tbl["doc_id"].to_pylist(), audit_tbl["verdict"].to_pylist()):
        if verdict == "drop":
            drop_ids.add(int(did))
print(f"🧹 Audit: bỏ qua {len(drop_ids):,} documents (verdict=drop)")

# Khởi tạo Chunker (750 chars chunk, 120 overlap - đảm bảo LCS/|g| >= 0.40 VÀ LCS/|c| >= 0.40)
chunker = DocumentChunker(
    max_chunk_size=750,
    chunk_overlap=120,
    min_chunk_size=50,
    split_by_sentences=True,
    enable_contextual=True,
    boilerplate=boilerplate,
)

cleaning_signature = f"boilerplate+audit-drop|{len(boilerplate)}domains|{len(drop_ids)}drops"

# Kết nối database
conn = sqlite3.connect(str(sqlite_path))
cur = conn.cursor()
cur.execute("PRAGMA synchronous = OFF;")
cur.execute("PRAGMA journal_mode = MEMORY;")

cur.execute("""
CREATE TABLE IF NOT EXISTS chunks (
    chunk_id INTEGER PRIMARY KEY,
    doc_id INTEGER,
    title TEXT,
    chunk_text TEXT,
    contextual_text TEXT,
    url TEXT,
    domain TEXT,
    lang TEXT,
    char_count INTEGER
);
""")
cur.execute("CREATE INDEX IF NOT EXISTS idx_chunks_doc_id ON chunks (doc_id);")
cur.execute("CREATE TABLE IF NOT EXISTS build_info (key TEXT PRIMARY KEY, value TEXT);")
conn.commit()

existing_chunks = cur.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
stored_sig = cur.execute("SELECT value FROM build_info WHERE key='cleaning'").fetchone()
if existing_chunks > 500000:
    if not stored_sig or stored_sig[0] != cleaning_signature:
        raise RuntimeError(
            "SQLite đang chứa chunks từ một build khác (chưa làm sạch hoặc khác bảng audit). "
            "Xóa data/indices/chunks_meta.sqlite rồi chạy lại cell này."
        )
    print(f"✓ SQLite đã có sẵn {existing_chunks:,} chunks sạch. Bỏ qua bước chunking!")
else:
    cur.execute("INSERT OR REPLACE INTO build_info VALUES ('cleaning', ?)", (cleaning_signature,))
    conn.commit()
    print("🚀 Bắt đầu quá trình cắt chunk 14 shards (Kích thước chuẩn 750 ký tự)...")
    global_chunk_id = 0
    skipped_docs = 0
    insert_buffer = []

    for shard_idx in range(14):
        shard_p = raw_corpus_dir / f"corpus_part_{shard_idx:03d}.parquet"
        if not shard_p.exists():
            continue

        tbl = pq.read_table(str(shard_p))
        doc_ids = tbl["doc_id"].to_pylist()
        urls = tbl["url"].to_pylist()
        titles = tbl["title"].to_pylist()
        texts = tbl["text"].to_pylist()
        langs = tbl["lang"].to_pylist()
        domains = tbl["domain"].to_pylist()

        for did, u, tit, txt, lg, dom in zip(doc_ids, urls, titles, texts, langs, domains):
            if not txt or len(txt.strip()) < 50:
                skipped_docs += 1
                continue
            if int(did) in drop_ids:
                skipped_docs += 1
                continue

            doc_chunks = chunker.chunk_document(
                doc_id=str(did),
                text=txt,
                metadata={"title": tit or "", "domain": dom or ""},
                lang=lg or "vi",
            )

            for ch in doc_chunks:
                insert_buffer.append((
                    global_chunk_id,
                    int(did),
                    tit or "",
                    ch.chunk_text,
                    ch.contextual_text or ch.chunk_text,
                    u or "",
                    dom or "",
                    ch.lang,
                    len(ch.chunk_text),
                ))
                global_chunk_id += 1

                if len(insert_buffer) >= 20000:
                    cur.executemany("""
                    INSERT INTO chunks (chunk_id, doc_id, title, chunk_text, contextual_text, url, domain, lang, char_count)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                    """, insert_buffer)
                    conn.commit()
                    insert_buffer.clear()

        print(f"  ✓ Shard {shard_idx:02d}/13: Đã xử lý {len(tbl):,} docs. Tổng chunks lũy kế: {global_chunk_id:,}")
        try:
            shard_p.unlink()
        except Exception:
            pass

    if insert_buffer:
        cur.executemany("""
        INSERT INTO chunks (chunk_id, doc_id, title, chunk_text, contextual_text, url, domain, lang, char_count)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, insert_buffer)
        conn.commit()
        insert_buffer.clear()

    total_chunks = cur.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
    print(f"\n🎉 ĐÃ CẮT XONG: {total_chunks:,} chunks (bỏ qua {skipped_docs:,} documents)")

conn.close()

In [ ]:
# 7. Mã hóa Dense Vector BGE-M3 trên GPU & Xây dựng FAISS Index (Có Auto-Resume Checkpoint)
import gc
import json
import sqlite3
import time
import numpy as np
import torch
import faiss
from FlagEmbedding import BGEM3FlagModel

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

num_gpus = torch.cuda.device_count()
print(f"Số lượng GPU khả dụng: {num_gpus}")
for g in range(num_gpus):
    print(f"  - GPU {g}: {torch.cuda.get_device_name(g)} ({torch.cuda.get_device_properties(g).total_memory / (1024**3):.1f} GB VRAM)")

conn = sqlite3.connect(str(sqlite_path))
cur = conn.cursor()
total_chunks = cur.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
print(f"Tổng số chunks cần mã hóa vector: {total_chunks:,}")

dimension = 1024
partial_faiss_path = indices_dir / "dense_index_partial.faiss"
final_faiss_path = indices_dir / "dense_index.faiss"

if partial_faiss_path.exists() and partial_faiss_path.stat().st_size > 1_000_000:
    index = faiss.read_index(str(partial_faiss_path))
    processed = index.ntotal
    print(f"🔄 Chạy tiếp từ checkpoint: {processed:,} / {total_chunks:,} vectors")
elif final_faiss_path.exists() and final_faiss_path.stat().st_size > 1_000_000:
    index = faiss.read_index(str(final_faiss_path), faiss.IO_FLAG_MMAP)
    processed = index.ntotal
    print(f"✓ Đã có FAISS index hoàn chỉnh ({processed:,} vectors). Bỏ qua bước mã hóa.")
else:
    index = faiss.IndexFlatIP(dimension)
    processed = 0

if processed < total_chunks:
    print("\nĐang tải BAAI/bge-m3 lên GPU (FP16)...")
    model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, device="cuda:0")

    batch_size = 64
    stream_batch_size = 50000
    t0 = time.time()
    step_processed = 0

    for offset in range(processed, total_chunks, stream_batch_size):
        cur.execute(f"""
        SELECT chunk_id, contextual_text
        FROM chunks
        ORDER BY chunk_id
        LIMIT {stream_batch_size} OFFSET {offset}
        """)
        rows = cur.fetchall()
        if not rows:
            break

        texts = [r[1] for r in rows]
        out = model.encode(
            texts,
            batch_size=batch_size,
            max_length=512,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
        dense_vecs = out["dense_vecs"].astype(np.float32)
        faiss.normalize_L2(dense_vecs)
        index.add(dense_vecs)

        processed += len(texts)
        step_processed += len(texts)
        elapsed = time.time() - t0
        rate = step_processed / elapsed if elapsed > 0 else 0
        eta_mins = (total_chunks - processed) / rate / 60 if rate > 0 else 0
        print(f"Đã mã hóa: {processed:,} / {total_chunks:,} chunks ({processed/total_chunks*100:.1f}%) | Tốc độ: {rate:.1f} chunks/s | Còn lại: {eta_mins:.1f} phút")

        if processed % 100000 < stream_batch_size or processed >= total_chunks:
            faiss.write_index(index, str(partial_faiss_path))
            print(f"  💾 Đã lưu Checkpoint ({index.ntotal:,} vectors) vào {partial_faiss_path}")

        del texts, out, dense_vecs
        torch.cuda.empty_cache()
        gc.collect()

conn.close()

if processed < total_chunks:
    raise RuntimeError(
        f"Mới mã hóa {processed:,}/{total_chunks:,} chunks. Chạy lại cell này để resume từ checkpoint."
    )

faiss_out = indices_dir / "dense_index.faiss"
if partial_faiss_path.exists():
    print(f"Đổi tên {partial_faiss_path.name} thành {faiss_out.name} (tiết kiệm 10GB đĩa)...")
    if faiss_out.exists():
        faiss_out.unlink()
    partial_faiss_path.rename(faiss_out)

n_vectors = index.ntotal
if n_vectors != total_chunks:
    print(f"⚠️ Lệch số vector ({n_vectors:,}) và số chunks ({total_chunks:,}) — kiểm tra lại checkpoint!")

meta_out = indices_dir / "dense_metadata.json"
with open(meta_out, "w", encoding="utf-8") as f:
    json.dump({
        "num_vectors": n_vectors,
        "num_chunks_in_sqlite": total_chunks,
        "dimension": dimension,
        "model_name": "BAAI/bge-m3",
        "cleaning": "boilerplate+audit-drop",
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }, f, indent=2)

size_mb = faiss_out.stat().st_size / (1024 * 1024)
print("=" * 60)
print("🎉 HOÀN TẤT DENSE INDEXING:")
print(f"  Tổng số vectors trong FAISS: {n_vectors:,}")
print(f"  Tổng chunks trong SQLite:    {total_chunks:,}")
print(f"  Dung lượng FAISS index:      {size_mb:.2f} MB")
print("=" * 60)

In [ ]:
# 8. Kiểm tra thử nghiệm truy hồi Dense Vector
if "model" not in globals():
    from FlagEmbedding import BGEM3FlagModel
    model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, device="cuda:0")

test_q = "viêm gan B lây qua đường nào"
q_out = model.encode([test_q], return_dense=True)["dense_vecs"].astype(np.float32)
faiss.normalize_L2(q_out)

scores, indices = index.search(q_out, 3)
print(f"Query: '{test_q}'")
for rank, (score, cid) in enumerate(zip(scores[0], indices[0]), 1):
    print(f"  Top {rank}: chunk_id={cid} (Cosine Similarity: {score:.4f})")

In [ ]:
# 9. Đồng bộ chunks_meta.sqlite và dense_index.faiss lên Hugging Face Bucket
import subprocess

print("📤 Đang upload chunks_meta.sqlite (mới) lên HF Bucket...")
!hf buckets cp data/indices/chunks_meta.sqlite hf://buckets/nieths/ViBioMIR/indices/chunks_meta.sqlite

print("📤 Đang upload dense_metadata.json lên HF Bucket...")
!hf buckets cp data/indices/dense_metadata.json hf://buckets/nieths/ViBioMIR/indices/dense_metadata.json

print("📤 Đang upload dense_index.faiss lên HF Bucket...")
!hf buckets cp data/indices/dense_index.faiss hf://buckets/nieths/ViBioMIR/indices/dense_index.faiss

print("\n" + "=" * 60)
print("🎉 TOÀN BỘ CHỈ MỤC MỚI (CHUNKS MỚI + DENSE VECTOR) ĐÃ ĐƯỢC ĐỒNG BỘ LÊN HF BUCKET!")
print("Hệ thống hiện đã phủ hơn 84% dữ liệu toàn cuộc thi (1,025,647 bài viết).")
print("Bây giờ bạn có thể mở notebook 'inference-queries.ipynb' để chạy ra file submission cuối cùng!")
print("=" * 60)